# Chapter 2 — First-Order Logic and Reasoning
### Notebook 0 · Overview and setup

*Book reference: Keet, *Ontology Engineering* (2nd ed.), Ch. 2*

Chapter 2 is the logical foundation the rest of the book stands on. Read passively it is a page of symbols. Here every piece of it is a running program: formulas are objects, models are data, and **M ⊨ φ** is a function call.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch02_toolkit as fol
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import oe_course; print(json.dumps(oe_course.describe_environment(), indent=1))

## Notebooks in this chapter

| # | Notebook | Book section | What you build |
|---|---|---|---|
| 0 | `00_overview_and_setup` | — | a working FOL engine |
| 1 | `01_syntax_and_semantics` | 2.1 | a parser, and Tarskian satisfaction as code |
| 2 | `02_reasoning` | 2.2 | entailment, countermodels, resolution proofs — and the decidability wall |
| 3 | `03_exercises` | 2.3 | autograded answers |
| 4 | `04_agentic_lab` | — | a formaliser agent, graded *semantically* |


**By the end of this notebook you can:**

1. Read and write FOL, and check your reading against a machine rather than against your intuition.
2. Construct a **countermodel** to show that two formalisations differ — the single most useful skill in this chapter.
3. Run a resolution refutation and read the resulting **proof**.
4. State precisely what finite model checking can and cannot decide.
5. Build an agent that formalises English, graded by semantics rather than string matching.

## The engine

`ch02_toolkit` is ~400 lines of pure Python: a tokeniser, a recursive-descent parser, an evaluator, a finite-model enumerator, and a resolution prover. No external solver. You can read all of it, and you should — the point of this chapter is that none of this is magic.

In [ ]:
f = fol.parse('forall x (Human(x) -> Mortal(x))')
print('AST      :', f)
print('rendered :', fol.to_string(f))
print('signature:', fol.predicates(f))

### Sanity check: the oldest argument in logic

In [ ]:
premises = [fol.parse('forall x (Human(x) -> Mortal(x))'),
            fol.parse('Human(Socrates)')]
conclusion = fol.parse('Mortal(Socrates)')
holds, countermodel = fol.entails(premises, conclusion, max_size=3)
print('entails:', holds, '| countermodel:', countermodel)
assert holds and countermodel is None